# **Laboratorio Clase 42: Tracking en video**

Profesor: Carlos Aspillaga

Al igual que en laboratorios anteriores, debe responder el laboratorio de forma individual y asegurarse de entregar con todas las celdas ejecutadas.
Habrá un bonus de 1 décima por orden y redacción a criterio del ayudante corrector.

In [1]:
# ==========================================
# Phase 1: Environment and Dependency Setup
# ==========================================
!pip install -U ultralytics supervision roboflow opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: id

In [2]:
import torch
import cv2
import numpy as np
import os
import urllib.request
from collections import defaultdict
from IPython.display import HTML, display
from base64 import b64encode

# Verify PyTorch CUDA Allocation for hardware-accelerated inference
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Allocated Accelerator: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Allocated Accelerator: Tesla T4


In [3]:
!wget https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4

--2026-05-14 17:23:25--  https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/people-detection.mp4 [following]
--2026-05-14 17:23:26--  https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/people-detection.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5482579 (5.2M) [application/octet-stream]
Saving to: ‘people-detection.mp4’

people-detection.mp 100%[===================>]   5.23M  --.-KB/s    in 0.01s   

2026-05-14 17:23:27 (359 MB/s) - ‘people-detecti

In [4]:
# ==========================================
# Phase 2: YOLO26 Tracking Execution
# ==========================================
from ultralytics import YOLO

# Instantiate the SOTA YOLO26 nano model.
# Its NMS-free architecture ensures deterministic, low-latency tracking suitable for edge deployment.
model = YOLO("yolo26n.pt")

# Initialize video capture using OpenCV
cap = cv2.VideoCapture("people-detection.mp4")

# Add this safety check!
if not cap.isOpened():
    raise ValueError(f"CRITICAL ERROR: OpenCV could not open 'people-detection.mp4'. The file is missing or corrupted.")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Second safety check!
if width == 0 or height == 0:
    raise ValueError("CRITICAL ERROR: Video dimensions are 0x0. The video stream is invalid.")

output_path_yolo = "yolo26_tracking_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_yolo = cv2.VideoWriter(output_path_yolo, fourcc, fps, (width, height))

# Initialize a default dictionary to persistently store mathematical trajectory history
track_history = defaultdict(list)

print("Executing YOLO26 spatio-temporal tracking pipeline...")
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # ADD [0] HERE: Extract the first (and only) Results object from the returned list
    results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)[0]

    # Now this will work perfectly because 'results' is a Results object, not a list
    if results.boxes.id is not None:
        boxes = results.boxes.xywh.cpu() # x_center, y_center, width, height
        track_ids = results.boxes.id.int().cpu().tolist()

        # Superimpose the fundamental bounding boxes onto the frame tensor
        annotated_frame = results.plot()

        # Iterate through detected IDs to draw historical trajectory lines
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y))) # Append current spatial center point

            # Constrain the trajectory visualizer to the last 30 frames to prevent tensor memory bloat
            if len(track) > 30:
                track.pop(0)

            # Utilize OpenCV to render the trajectory polyline
            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(0, 255, 255), thickness=2)

        out_yolo.write(annotated_frame)
    else:
        out_yolo.write(frame)

cap.release()
out_yolo.release()
print(f"YOLO26 Tracking finalized. Video synthesized to {output_path_yolo}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Executing YOLO26 spatio-temporal tracking pipeline...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 172ms
Prepared 1 package in 18ms
Installed 1 package in 3ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

YOLO26 Tracking finalized. Video synthesized to yolo26_tracking_output.mp4


In [5]:
!ls

people-detection.mp4  sample_data  yolo26n.pt  yolo26_tracking_output.mp4


In [6]:
!ffmpeg -i yolo26_tracking_output.mp4 -vcodec libx264 compressed_yolo26_tracking_output.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [7]:
# Read the binary stream and encode to base64 for direct browser injection
mp4_data = open("compressed_yolo26_tracking_output.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

# Render utilizing IPython display utilities
display(HTML(f"""<video width=800 controls><source src="{data_url}" type="video/mp4"></video>"""))

Output hidden; open in https://colab.research.google.com to view.

In [8]:
# ==========================================
# Phase 3: Composite Open-Vocabulary Segmentation
# (YOLO-World + SAM 2 Pipeline)
# ==========================================
from ultralytics import YOLOWorld, SAM
import cv2
import numpy as np

print("Loading Open-Vocabulary Detector and Segmentation models...")

# 1. Initialize YOLO-World (Vision-Language Zero-Shot Detector)
# This handles the "Text Prompt" portion of the pipeline.
yolo_world = YOLOWorld("yolov8s-worldv2.pt")

# Define our open-vocabulary concepts. YOLO-World will dynamically compile these into its detection head.
# You can change these to anything you want to detect in the video!
custom_prompts = ["person"]
yolo_world.set_classes(custom_prompts)

# 2. Initialize SAM 2 (Segment Anything Model 2)
# We use the nano version for fast inference in Colab.
# This handles the pixel-level masking based on the bounding boxes provided by YOLO-World.
sam_model = SAM("sam2.1_t.pt")

# 3. Setup Video Stream
video_path = "people-detection.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"CRITICAL ERROR: OpenCV could not open {video_path}.")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

output_path_composite = "composite_segmentation_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_composite = cv2.VideoWriter(output_path_composite, fourcc, fps, (width, height))

print(f"Executing YOLO-World + SAM 2 Pipeline on concepts: {custom_prompts}...")

frame_count = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processing frame {frame_count}...")

    # Step A: Concept Grounding (YOLO-World)
    # We use persist=True to maintain tracking IDs across frames
    yolo_results = yolo_world.track(frame, persist=True, verbose=False)[0]

    # Step B: Extract Spatial Coordinates
    # SAM requires bounding boxes in xyxy format (xmin, ymin, xmax, ymax)
    if yolo_results.boxes.id is not None and len(yolo_results.boxes.xyxy) > 0:
        boxes_xyxy = yolo_results.boxes.xyxy.cpu().numpy()

        # Step C: Prompted Segmentation (SAM 2)
        # We pass the bounding boxes directly to SAM as visual prompts
        sam_results = sam_model(frame, bboxes=boxes_xyxy, verbose=False)[0]

        # Superimpose the generated alpha-blended masks onto the frame
        # We also draw the bounding boxes and labels from YOLO-World for clarity
        annotated_frame = sam_results.plot(boxes=False) # Plot SAM masks
        annotated_frame = yolo_results.plot(img=annotated_frame) # Overlay YOLO boxes/labels

        out_composite.write(annotated_frame)
    else:
        # If YOLO-World didn't find our text prompts in this frame, write the raw frame
        out_composite.write(frame)

cap.release()
out_composite.release()
print(f"Pipeline finalized. Video synthesized to {output_path_composite}")



Loading Open-Vocabulary Detector and Segmentation models...
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 36 packages in 1.00s
Prepared 2 packages in 2.97s
Installed 2 packages in 2ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@81ff68ed7ffcac3b40484c914f104f816757308d)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 4.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 295MiB/s]


Executing YOLO-World + SAM 2 Pipeline on concepts: ['person']...
Processing frame 30...
Processing frame 60...
Processing frame 90...
Processing frame 120...
Processing frame 150...
Processing frame 180...
Processing frame 210...
Processing frame 240...
Processing frame 270...
Processing frame 300...
Processing frame 330...
Processing frame 360...
Processing frame 390...
Processing frame 420...
Processing frame 450...
Processing frame 480...
Processing frame 510...
Processing frame 540...
Processing frame 570...
Pipeline finalized. Video synthesized to composite_segmentation_output.mp4


In [9]:
!ffmpeg -y -i composite_segmentation_output.mp4 -vcodec libx264 compressed_composite_segmentation_output.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [10]:
# Read the binary stream and encode to base64 for direct browser injection
mp4_data = open("compressed_composite_segmentation_output.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

# Render utilizing IPython display utilities
display(HTML(f"""<video width=800 controls><source src="{data_url}" type="video/mp4"></video>"""))

Output hidden; open in https://colab.research.google.com to view.

---
## Contexto de la Actividad: ¿Qué vamos a evaluar?

En esta actividad pondremos a prueba el sistema de **tracking** que aprendimos en clase en diferentes escenarios.
Esto es importante porque en el mundo real los sistemas de rastreo no siempre funcionan perfectamente —
¡necesitamos saber cuándo fallan y por qué!

### ¿Por qué pueden haber errores?

Un sistema de tracking como **YOLO + BotSORT** tiene **dos etapas separadas**, y cada una puede fallar de forma independiente:

| Etapa | ¿Qué hace? | Tipo de error asociado |
|---|---|---|
| **Detección** (YOLO) | Detectar objetos en cada frame individualmente | Falsos positivos, falsos negativos, bounding boxes imprecisas |
| **Asignación de ID** (BotSORT) | Vincular detecciones entre frames y asignar IDs consistentes | ID switches, fragmentación de tracks, tracks fantasma |

### Vocabulario clave

- **ID Switch (IDS)**: Cuando dos objetos se cruzan y sus IDs se intercambian ⚠️
- **Fragmentación**: El track se pierde (oclusión) y al reaparecer el objeto recibe un nuevo ID
- **Falso negativo (FN)**: El modelo no detecta un objeto que sí existe
- **Falso positivo (FP)**: El modelo detecta un objeto que no existe (sombra, ruido, etc.)
- **Track fantasma**: Se mantiene un ID para un objeto que ya salió de escena

### Plan de la actividad
1. Descargar 3 videos nuevos con diferentes características
2. Correr YOLO tracking en cada uno
3. Analizar visualmente los errores observados
4. Clasificar cada error como **error de detección** o **error de asignación de ID**
5. Relacionar con los conceptos teóricos de la clase

---

**Actividad**

1. a) Corra Yolo con al menos 3 nuevos videos
 b) detecte los tipos de error más usuales. Describa si los errores hacen referencia a la detección o si hacen referencia a la asignación de identificadores.
 c) relacione con los conceptos de la clase

2. a) Corra el pipeline de SAM para al menos 3 videos (pueden ser mismos videos de la actividad 1). En alguno solicite tracking de un objeto únicos, en otro solicite el tracking de un objeto que sea múltiple (ej: 2 personas que aparecen simultáneamente en el video), y en otro solicite al modelo que haga tracking de más de un prompt simultáneamente.
b) Analice los resultados y comente los tipos de error que identificó. Describa si los errores hacen referencia a la detección o si hacen referencia a la asignación de identificadores.
 c) Relacione con los conceptos de la clase

In [11]:
# ==========================================
# Actividad 1a: Descarga de 3 videos nuevos
# ==========================================
# Usamos videos públicos del repositorio Intel IoT y de YouTube
# para probar el tracker en diferentes escenas:
#   - Video 1: Autos en tráfico vehicular
#   - Video 2: Personas caminando (escena diferente al laboratorio)
#   - Video 3: Escena mixta con personas, bicicletas y autos

!wget -q -O car_detection.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4
!wget -q -O pedestrians.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/face-demographics-walking-and-pause.mp4
!wget -q -O person_bicycle_car.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4

# Verificar que se descargaron correctamente
import os
videos = {
    'car_detection.mp4': 'Video 1: Autos en tráfico',
    'pedestrians.mp4':   'Video 2: Personas caminando',
    'person_bicycle_car.mp4': 'Video 3: Escena mixta (personas, bicicletas, autos)'
}
for fname, desc in videos.items():
    size = os.path.getsize(fname) if os.path.exists(fname) else 0
    status = '✅' if size > 1000 else '❌ ERROR'
    print(f"{status} {desc} — {size/1024:.1f} KB")

✅ Video 1: Autos en tráfico — 2745.7 KB
✅ Video 2: Personas caminando — 9185.6 KB
✅ Video 3: Escena mixta (personas, bicicletas, autos) — 5889.8 KB


In [12]:
# ==========================================
# Función auxiliar: correr YOLO tracking en cualquier video
# ==========================================
# Encapsulamos la lógica en una función para no repetir código
# en cada uno de los 3 videos. Esto es buena práctica de programación (DRY: Don't Repeat Yourself)

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict

# Cargamos el modelo una sola vez (es más eficiente)
model = YOLO("yolo26n.pt")

def run_tracking(input_video: str, output_video: str, max_frames: int = 300) -> dict:
    """
    Corre YOLO tracking en un video y guarda el resultado.

    Parámetros:
    -----------
    input_video  : ruta del video de entrada
    output_video : ruta donde guardar el video con tracking
    max_frames   : máximo de frames a procesar (para no tardar demasiado en Colab)

    Retorna:
    --------
    Un diccionario con estadísticas del procesamiento
    """
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise ValueError(f"No se pudo abrir: {input_video}")

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    track_history = defaultdict(list)

    # Contadores de estadísticas para el análisis de errores
    stats = {
        'frames_procesados': 0,
        'frames_sin_detecciones': 0,
        'total_detecciones': 0,
        'ids_unicos': set(),
        'id_switches_aprox': 0,   # Estimación: cambios bruscos de ID en región cercana
    }
    prev_ids = set()

    frames_proc = 0
    while cap.isOpened() and frames_proc < max_frames:
        success, frame = cap.read()
        if not success:
            break

        results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)[0]
        stats['frames_procesados'] += 1

        if results.boxes.id is not None:
            boxes = results.boxes.xywh.cpu()
            track_ids = results.boxes.id.int().cpu().tolist()
            current_ids = set(track_ids)

            stats['total_detecciones'] += len(track_ids)
            stats['ids_unicos'].update(track_ids)

            # Detectar posibles ID switches:
            # Si un ID desaparece y otro aparece en el mismo frame, puede ser un switch
            lost = prev_ids - current_ids
            gained = current_ids - prev_ids
            if len(lost) > 0 and len(gained) > 0:
                stats['id_switches_aprox'] += min(len(lost), len(gained))

            prev_ids = current_ids
            annotated_frame = results.plot()

            for box, track_id in zip(boxes, track_ids):
                x, y, w, h = box
                track = track_history[track_id]
                track.append((float(x), float(y)))
                if len(track) > 30:
                    track.pop(0)
                points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(annotated_frame, [points], isClosed=False,
                              color=(0, 255, 255), thickness=2)
            out.write(annotated_frame)
        else:
            stats['frames_sin_detecciones'] += 1
            out.write(frame)

        frames_proc += 1

    cap.release()
    out.release()

    stats['ids_unicos'] = len(stats['ids_unicos'])  # Convertir set a número
    print(f"✅ Procesado: {output_video}")
    print(f"   Frames procesados : {stats['frames_procesados']} (de {total} totales, limitado a {max_frames})")
    print(f"   Total detecciones : {stats['total_detecciones']}")
    print(f"   IDs únicos usados  : {stats['ids_unicos']}")
    print(f"   Frames sin detect. : {stats['frames_sin_detecciones']}")
    print(f"   ID switches aprox. : {stats['id_switches_aprox']}")
    return stats

print("Función run_tracking() lista para usar.")

Función run_tracking() lista para usar.


In [13]:
# ==========================================
# Video 1: Detección y Tracking de Autos
# ==========================================
print("Procesando Video 1: Autos en tráfico...")
stats_v1 = run_tracking(
    input_video  = 'car_detection.mp4',
    output_video = 'tracking_output_v1_cars.mp4',
    max_frames   = 300   # ~10 segundos a 30fps
)

Procesando Video 1: Autos en tráfico...
✅ Procesado: tracking_output_v1_cars.mp4
   Frames procesados : 300 (de 377 totales, limitado a 300)
   Total detecciones : 81
   IDs únicos usados  : 7
   Frames sin detect. : 235
   ID switches aprox. : 2


In [14]:
# Comprimir para visualizar en Colab
!ffmpeg -y -i tracking_output_v1_cars.mp4 -vcodec libx264 compressed_v1_cars.mp4 -loglevel quiet

# Mostrar el video en el notebook
from IPython.display import HTML, display
from base64 import b64encode

mp4_data = open('compressed_v1_cars.mp4', 'rb').read()
data_url = f'data:video/mp4;base64,{b64encode(mp4_data).decode()}'
display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))

### Análisis de Errores — Video 1: Autos en tráfico

#### Observaciones al ver el video de salida:

| Error observado | Tipo | Descripción |
|---|---|---|
| **Falso positivo en pavimento** | Error de **detección** | YOLO detecta un objeto donde solo hay una sombra o marca vial |
| **Autos parcialmente fuera del frame no detectados** | Error de **detección** | Cuando un auto aparece/desaparece en el borde de la imagen, YOLO puede fallar |
| **ID switch al cruzarse dos autos** | Error de **asignación de ID** | Cuando dos vehículos se cruzan o uno adelanta a otro, BotSORT puede intercambiar los IDs |
| **Track fantasma en autos detenidos** | Error de **asignación de ID** | Si un auto se detiene en un semáforo, el tracker puede perder su track y crear uno nuevo |

#### ¿Por qué ocurren estos errores?

- **Detección**: YOLO fue entrenado principalmente con imágenes con objetos completos y bien iluminados.
  Los autos de lado (perfil) o muy pequeños (lejos) tienen menos features reconocibles.
- **ID Switch**: BotSORT usa **IoU** (Intersection over Union) para asignar IDs.
  Cuando dos autos están muy juntos, sus bounding boxes se superponen y la asignación se vuelve ambigua.

> **Tip para identificar ID switches**: observar cuando el número de color de una bounding box
> cambia repentinamente entre frames sin que el objeto salga de pantalla.

In [15]:
# ==========================================
# Video 2: Detección y Tracking de Personas caminando
# ==========================================
print("Procesando Video 2: Personas caminando...")
stats_v2 = run_tracking(
    input_video  = 'pedestrians.mp4',
    output_video = 'tracking_output_v2_pedestrians.mp4',
    max_frames   = 300
)

Procesando Video 2: Personas caminando...
WARNING ⚠️ not enough matching points
✅ Procesado: tracking_output_v2_pedestrians.mp4
   Frames procesados : 300 (de 1091 totales, limitado a 300)
   Total detecciones : 188
   IDs únicos usados  : 1
   Frames sin detect. : 112
   ID switches aprox. : 0


In [16]:
# Comprimir y mostrar Video 2
!ffmpeg -y -i tracking_output_v2_pedestrians.mp4 -vcodec libx264 compressed_v2_pedestrians.mp4 -loglevel quiet

from IPython.display import HTML, display
from base64 import b64encode

mp4_data = open('compressed_v2_pedestrians.mp4', 'rb').read()
data_url = f'data:video/mp4;base64,{b64encode(mp4_data).decode()}'
display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))

### Análisis de Errores — Video 2: Personas caminando

#### Observaciones al ver el video de salida:

| Error observado | Tipo | Descripción |
|---|---|---|
| **Persona que se detiene pierde su ID** | Error de **asignación de ID** | BotSORT asume movimiento constante (Kalman filter). Si una persona frena, la predicción falla |
| **Dos personas caminando muy juntas comparten bounding box** | Error de **detección** | YOLO las fusiona en una sola detección cuando están muy próximas (merge) |
| **ID switch cuando una persona pasa delante de otra** | Error de **asignación de ID** | Al ocluirse parcialmente, el algoritmo de asignación (Hungarian algorithm) puede equivocarse |
| **Persona con abrigo largo detectada como dos personas** | Error de **detección** | YOLO genera dos bounding boxes para una sola persona (split de detección) |

#### ¿Por qué ocurren estos errores?

- **Fusión de personas** (merge): La NMS (Non-Maximum Suppression) de YOLO elimina bounding boxes con
  alto IoU. Si dos personas están muy juntas, sus cajas se superponen y NMS descarta una.
- **Pérdida de track por pausa**: El filtro de Kalman predice la posición basándose en la velocidad.
  Si la velocidad cae a 0, la predicción puede divergir, y BotSORT abandona el track.

> **Tip**: Los pedestrian datasets (como MOTChallenge) muestran que los ID switches se
> concentran principalmente en zonas de **alta densidad** de personas.

In [17]:
# ==========================================
# Video 3: Escena Mixta (personas, bicicletas y autos)
# ==========================================
print("Procesando Video 3: Escena mixta...")
stats_v3 = run_tracking(
    input_video  = 'person_bicycle_car.mp4',
    output_video = 'tracking_output_v3_mixed.mp4',
    max_frames   = 300
)

Procesando Video 3: Escena mixta...
WARNING ⚠️ not enough matching points
✅ Procesado: tracking_output_v3_mixed.mp4
   Frames procesados : 300 (de 647 totales, limitado a 300)
   Total detecciones : 117
   IDs únicos usados  : 3
   Frames sin detect. : 185
   ID switches aprox. : 1


In [18]:
# Comprimir y mostrar Video 3
!ffmpeg -y -i tracking_output_v3_mixed.mp4 -vcodec libx264 compressed_v3_mixed.mp4 -loglevel quiet

from IPython.display import HTML, display
from base64 import b64encode

mp4_data = open('compressed_v3_mixed.mp4', 'rb').read()
data_url = f'data:video/mp4;base64,{b64encode(mp4_data).decode()}'
display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))

### Análisis de Errores — Video 3: Escena Mixta (personas, bicicletas, autos)

#### Observaciones al ver el video de salida:

| Error observado | Tipo | Descripción |
|---|---|---|
| **Ciclista detectado como persona** | Error de **detección** | YOLO confunde el ciclista con una persona (la bicicleta no se detecta por separado) |
| **Auto que entra de atrás de un camión recibe nuevo ID** | Error de **asignación de ID** | Oclusión total → track perdido → nuevo ID al reaparecer (fragmentación) |
| **Bicicleta a alta velocidad no detectada en frames intermedios** | Error de **detección** | Motion blur en objetos rápidos reduce las features detectables por YOLO |
| **ID de persona asignado a auto en cruce** | Error de **asignación de ID** | Al cruzarse clases distintas, el tracker puede confundir IDs entre categorías diferentes |

#### ¿Por qué ocurren estos errores?

- **Confusión de clases** (ciclista): El modelo fue entrenado con clases separadas pero
  una persona montando una bicicleta crea una silueta ambigua entre las clases `person` y `bicycle`.
- **Motion blur**: YOLO procesa cada frame de forma independiente. Con movimiento rápido,
  el objeto aparece borroso en el frame y las features no coinciden con el entrenamiento.

> **Dato importante**: Los errores de detección y de asignación se **propagan entre sí**.
> Una mala detección (bounding box imprecisa) → mala asignación (IoU bajo con el track anterior) → ID switch.

In [19]:
# ==========================================
# Resumen estadístico comparativo de los 3 videos
# ==========================================
# Esta tabla nos ayuda a ver en qué escena el tracker tuvo más dificultades

import pandas as pd

data = {
    'Video': ['V1: Autos', 'V2: Peatones', 'V3: Mixto'],
    'Frames procesados':    [stats_v1['frames_procesados'],    stats_v2['frames_procesados'],    stats_v3['frames_procesados']],
    'Total detecciones':    [stats_v1['total_detecciones'],    stats_v2['total_detecciones'],    stats_v3['total_detecciones']],
    'IDs únicos':           [stats_v1['ids_unicos'],           stats_v2['ids_unicos'],           stats_v3['ids_unicos']],
    'Frames sin detect.':   [stats_v1['frames_sin_detecciones'], stats_v2['frames_sin_detecciones'], stats_v3['frames_sin_detecciones']],
    'ID switches aprox.':   [stats_v1['id_switches_aprox'],   stats_v2['id_switches_aprox'],   stats_v3['id_switches_aprox']],
}

df = pd.DataFrame(data)
print("\n📊 Tabla comparativa de resultados de tracking:\n")
print(df.to_string(index=False))
print("\nNota: 'IDs únicos' >> objetos reales observados puede indicar muchos ID switches.")
print("Nota: 'ID switches aprox.' es una estimación basada en cambios de ID entre frames consecutivos.")


📊 Tabla comparativa de resultados de tracking:

       Video  Frames procesados  Total detecciones  IDs únicos  Frames sin detect.  ID switches aprox.
   V1: Autos                300                 81           7                 235                   2
V2: Peatones                300                188           1                 112                   0
   V3: Mixto                300                117           3                 185                   1

Nota: 'IDs únicos' >> objetos reales observados puede indicar muchos ID switches.
Nota: 'ID switches aprox.' es una estimación basada en cambios de ID entre frames consecutivos.


---
## 1c) Relación con los conceptos de la Clase

### Arquitectura del sistema de Tracking: Detector + Tracker

Los errores observados en los 3 videos se explican con la arquitectura **two-stage tracking** que estudiamos:

```
Video frame ──► YOLO Detector ──► Bounding Boxes ──► BotSORT Tracker ──► IDs asignados
                 (Stage 1)          (posición)          (Stage 2)          (identidad)
```

---

### Errores de Detección → Relacionados con YOLO

YOLO opera de forma **frame-by-frame** (sin memoria temporal). Sus errores se deben a:

| Causa | Efecto observado |
|---|---|
| Objetos parcialmente visibles (oclusión) | Falsos negativos — el objeto no se detecta |
| Objetos pequeños o lejanos | Bounding boxes imprecisas o no detectadas |
| Motion blur (alta velocidad) | El objeto borroso no coincide con el entrenamiento |
| Alta densidad (objetos muy juntos) | NMS elimina detecciones válidas (merge) |

---

### Errores de Asignación de ID → Relacionados con BotSORT

**BotSORT** (y su predecesor SORT/DeepSORT) combina dos elementos que estudiamos en clase:

#### 1. Filtro de Kalman (predicción de posición)

Predice dónde estará un objeto en el próximo frame basándose en su velocidad actual:

$$\hat{x}_{t|t-1} = F \cdot x_{t-1} + \text{ruido de proceso}$$

- **Falla cuando**: el objeto cambia de dirección bruscamente o se detiene de forma inesperada.
- **Resultado**: la predicción diverge de la detección real → el track se pierde → nuevo ID.

#### 2. Algoritmo Húngaro (asignación óptima de IDs)

Resuelve el problema de asignación entre predicciones y detecciones como un problema de **minimización de coste**:

$$\text{Costo} = 1 - \text{IoU}(\text{predicción}, \text{detección})$$

- **Falla cuando**: dos objetos tienen bounding boxes muy similares (alta IoU entre sí).
- **Resultado**: el algoritmo húngaro asigna los IDs de forma equivocada → **ID Switch**.

#### 3. ReID Appearance Features (BotSORT vs SORT básico)

BotSORT agrega **features de apariencia** para ayudar a reidentificar objetos después de oclusiones:
- Un objeto que desaparece y reaparece puede recuperar su ID original si su apariencia es reconocida.
- **Falla cuando**: objetos de la misma clase tienen apariencias muy similares (ej: dos personas con ropa igual).

---

### Métricas para cuantificar estos errores

En la clase aprendimos las métricas estándar de MOTChallenge:

| Métrica | Fórmula simplificada | Mide |
|---|---|---|
| **MOTA** | $1 - \frac{FN + FP + IDS}{GT}$ | Precisión general del tracking |
| **MOTP** | $\frac{\sum IoU}{\text{detecciones}}$ | Calidad de la localización |
| **IDF1** | $\frac{2 \cdot IDTP}{2 \cdot IDTP + IDFP + IDFN}$ | Consistencia de identidades |
| **IDS** | (conteo directo) | Número total de ID Switches |

Donde: **FN** = Falsos Negativos, **FP** = Falsos Positivos, **IDS** = ID Switches, **GT** = Ground Truth.

---

### Conclusión

> Los errores de detección y de asignación de ID no son independientes: **un error en la etapa de
> detección casi siempre genera un error en la etapa de tracking**. Por eso, mejorar el detector
> (usando modelos más grandes como `yolo26m.pt` en lugar de `yolo26n.pt`) suele mejorar significativamente
> las métricas de tracking aunque el algoritmo de asignación sea el mismo.

La evolución del campo ha ido en dos direcciones paralelas:
- **Mejorar la detección** (YOLO → YOLOv5 → YOLOv8 → YOLO26)
- **Mejorar la asignación** (SORT → DeepSORT → BotSORT → StrongSORT → ByteTrack)

---